# 01 — Dense document embeddings for MS MARCO passage (Kaggle GPU)

Encodes all 8,841,823 passages with `BAAI/bge-small-en-v1.5` (384-d, L2-normalised) for the
dense arms (A5/A6). Run on Kaggle with **GPU T4 x2** and **Internet on**, via
*Save & Run All*. Output: `/kaggle/working/dense/shard_XXXX.npz` holding `doc_ids` and `emb`
(float16, about 6.8 GB in total). Resumable in the same way as notebook 00.

Passages are encoded **without** an instruction prefix. At query time the bge query
instruction (`QUERY_INSTRUCTION` below) is prepended, as the model card prescribes for retrieval.

In [ ]:
!pip install -q ir_datasets

In [ ]:
import json, shutil, time
from pathlib import Path

import ir_datasets
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL = "BAAI/bge-small-en-v1.5"
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
MAX_LEN = 256
BATCH = 512
SHARD_SIZE = 500_000
OUT = Path("/kaggle/working/dense")
RESUME_FROM = None
OUT.mkdir(parents=True, exist_ok=True)
if RESUME_FROM is not None:
    for f in RESUME_FROM.glob("shard_*.npz"):
        if not (OUT / f.name).exists():
            shutil.copy(f, OUT / f.name)
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
print(devices)

In [ ]:
model = SentenceTransformer(MODEL, device="cuda")
model.max_seq_length = MAX_LEN
model.half()
pool = model.start_multi_process_pool(target_devices=devices) if len(devices) > 1 else None

dataset = ir_datasets.load("msmarco-passage")
n_docs = dataset.docs_count()
n_shards = (n_docs + SHARD_SIZE - 1) // SHARD_SIZE
print(n_docs, "passages ->", n_shards, "shards")

In [ ]:
log = []
docs = iter(dataset.docs_iter())
for s in range(n_shards):
    path = OUT / f"shard_{s:04d}.npz"
    batch = [next(docs) for _ in range(min(SHARD_SIZE, n_docs - s * SHARD_SIZE))]
    if path.exists():
        continue
    t0 = time.time()
    texts = [d.text for d in batch]
    if pool is not None:
        emb = model.encode(texts, pool=pool, batch_size=BATCH, normalize_embeddings=True)
    else:
        emb = model.encode(texts, batch_size=BATCH, normalize_embeddings=True)
    np.savez(path, doc_ids=np.array([d.doc_id for d in batch]), emb=emb.astype(np.float16))
    log.append({"shard": s, "docs": len(batch), "seconds": time.time() - t0})
    print(log[-1], flush=True)

In [ ]:
if pool is not None:
    model.stop_multi_process_pool(pool)
meta = {"model": MODEL, "query_instruction": QUERY_INSTRUCTION, "max_len": MAX_LEN, "dim": model.get_sentence_embedding_dimension(),
        "normalized": True, "n_docs": n_docs, "shard_size": SHARD_SIZE, "log": log}
(OUT / "meta.json").write_text(json.dumps(meta, indent=2))
print(sum(f.stat().st_size for f in OUT.glob("*.npz")) / 1e9, "GB in", len(list(OUT.glob("*.npz"))), "shards")